In [1]:
import os
import requests

# Base URL of Argo data index files
base_url = "https://data-argo.ifremer.fr/"

# List of main index files to download
files_to_download = [
    "ar_index_global_meta.txt.gz",
    "ar_index_global_prof.txt.gz",
    "ar_index_global_tech.txt.gz",
    "ar_index_global_traj.txt.gz",
    "ar_index_this_week_meta.txt",
    "ar_index_this_week_prof.txt",
    "argo_bio-profile_index.txt.gz",
    "argo_bio-traj_index.txt.gz",
    "argo_synthetic-profile_index.txt.gz"
]

# Folder to save downloads
download_folder = "argo_index_files"
os.makedirs(download_folder, exist_ok=True)

# Function to download a single file with retries
def download_file(url, local_path, retries=3):
    for attempt in range(retries):
        try:
            print(f"Downloading {url} ...")
            r = requests.get(url, stream=True, timeout=60)
            r.raise_for_status()
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
            print(f"Downloaded: {local_path}")
            return True
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
    print(f"Failed to download {url} after {retries} attempts.")
    return False

# Loop through all files
for file_name in files_to_download:
    file_url = base_url + file_name
    local_path = os.path.join(download_folder, file_name)

    # Skip if already exists
    if os.path.exists(local_path):
        print(f"Already exists: {file_name}, skipping.")
        continue

    download_file(file_url, local_path)


Downloaded: argo_index_files/ar_index_global_meta.txt.gz
Downloaded: argo_index_files/ar_index_global_prof.txt.gz
Downloaded: argo_index_files/ar_index_global_tech.txt.gz
Downloaded: argo_index_files/ar_index_global_traj.txt.gz
Downloaded: argo_index_files/ar_index_this_week_meta.txt
Downloaded: argo_index_files/ar_index_this_week_prof.txt
Downloaded: argo_index_files/argo_bio-profile_index.txt.gz
Downloaded: argo_index_files/argo_bio-traj_index.txt.gz
Downloaded: argo_index_files/argo_synthetic-profile_index.txt.gz


In [2]:
import pandas as pd
import gzip
import os

gz_folder = "argo_index_files"
csv_folder = "index_files_csv"
os.makedirs(csv_folder, exist_ok=True)

for gz_file in os.listdir(gz_folder):
    if gz_file.endswith(".gz"):
        gz_path = os.path.join(gz_folder, gz_file)
        lines = []

        with gzip.open(gz_path, "rt") as f:
            for line in f:
                # Skip metadata/comment lines starting with #
                if line.startswith("#") or not line.strip():
                    continue
                # Split line by comma (assuming CSV format)
                parts = line.strip().split(",")
                lines.append(parts)

        # Create DataFrame using first row as column names
        if lines:
            df = pd.DataFrame(lines[1:], columns=lines[0])
        else:
            df = pd.DataFrame()  # empty file

        # Save CSV
        csv_file = os.path.join(csv_folder, gz_file.replace(".txt.gz", ".csv"))
        df.to_csv(csv_file, index=False)
        print(f"Converted {gz_file} → {csv_file}")


Converted argo_bio-profile_index.txt.gz → index_files_csv/argo_bio-profile_index.csv
Converted ar_index_global_traj.txt.gz → index_files_csv/ar_index_global_traj.csv
Converted ar_index_global_prof.txt.gz → index_files_csv/ar_index_global_prof.csv
Converted argo_bio-traj_index.txt.gz → index_files_csv/argo_bio-traj_index.csv
Converted ar_index_global_tech.txt.gz → index_files_csv/ar_index_global_tech.csv
Converted ar_index_global_meta.txt.gz → index_files_csv/ar_index_global_meta.csv
Converted argo_synthetic-profile_index.txt.gz → index_files_csv/argo_synthetic-profile_index.csv


In [3]:
import pandas as pd
import os # Import the os module

path = "/content/index_files_csv/"
for file_name in os.listdir(path): # Iterate over the list of filenames
    file_path = os.path.join(path, file_name) # Get the full file path
    print(f"Processing file: {file_path}")
    df = pd.read_csv(file_path)
    display(df.head())
    print()

Processing file: /content/index_files_csv/argo_bio-profile_index.csv


,file,date,latitude,longitude,ocean,profiler_type,institution,parameters,parameter_data_mode,date_update
0,aoml/1900722/profiles/BD1900722_001.nc,2.006102e+13,-40.316,73.389,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,20200312153230
1,aoml/1900722/profiles/BD1900722_002.nc,2.006110e+13,-40.390,73.528,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,20200312153230
2,aoml/1900722/profiles/BD1900722_003.nc,2.006111e+13,-40.455,73.335,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,20200312153230
3,aoml/1900722/profiles/BD1900722_004.nc,2.006112e+13,-40.134,73.080,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,20200312153230
4,aoml/1900722/profiles/BD1900722_005.nc,2.006120e+13,-39.641,73.158,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,20200312153230



Processing file: /content/index_files_csv/ar_index_global_prof.csv


,file,date,latitude,longitude,ocean,profiler_type,institution,date_update
0,aoml/13857/profiles/R13857_001.nc,1.997073e+13,0.267,-16.032,A,845,AO,20181011180520
1,aoml/13857/profiles/R13857_002.nc,1.997081e+13,0.072,-17.659,A,845,AO,20181011180521
2,aoml/13857/profiles/R13857_003.nc,1.997082e+13,0.543,-19.622,A,845,AO,20181011180521
3,aoml/13857/profiles/R13857_004.nc,1.997083e+13,1.256,-20.521,A,845,AO,20181011180521
4,aoml/13857/profiles/R13857_005.nc,1.997091e+13,0.720,-20.768,A,845,AO,20181011180521



Processing file: /content/index_files_csv/ar_index_global_traj.csv


...

,file,latitude_max,latitude_min,longitude_max,longitude_min,profiler_type,institution,parameters,date_update
0,bodc/6904189/6904189_BRtraj.nc,NaN,NaN,NaN,NaN,836,BO,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,20240110013018
1,csio/2901550/2901550_BRtraj.nc,36.813,27.921,155.622,134.477,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,20241116060644
2,csio/2901551/2901551_BRtraj.nc,29.891,23.523,149.631,139.986,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,20241116060654
3,csio/2901552/2901552_BRtraj.nc,30.627,25.121,149.524,140.608,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,20241116060700
4,csio/2901553/2901553_BRtraj.nc,36.322,27.627,162.148,140.424,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,20241116060710



Processing file: /content/index_files_csv/ar_index_global_meta.csv


,file,profiler_type,institution,date_update
0,aoml/13857/13857_meta.nc,845.0,AO,20181011200014
1,aoml/13858/13858_meta.nc,845.0,AO,20181011200015
2,aoml/13859/13859_meta.nc,845.0,AO,20181011200025
3,aoml/15819/15819_meta.nc,845.0,AO,20181011200016
4,aoml/15820/15820_meta.nc,845.0,AO,20181011200018



Processing file: /content/index_files_csv/ar_index_global_tech.csv


,file,institution,date_update
0,aoml/13857/13857_tech.nc,AO,20210428200335
1,aoml/13858/13858_tech.nc,AO,20210428200337
2,aoml/13859/13859_tech.nc,AO,20181011200025
3,aoml/15819/15819_tech.nc,AO,20210428200342
4,aoml/15820/15820_tech.nc,AO,20210428200346


In [4]:
import pandas as pd
import os

path = "/content/index_files_csv/"
for file_name in os.listdir(path):
    file_path = os.path.join(path, file_name)
    print(f"Processing file: {file_path}")
    df = pd.read_csv(file_path)

    # Convert columns that likely contain date information to datetime, coercing errors
    for col in df.columns:
        if 'date' in col.lower() or 'update' in col.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"Converted column '{col}' to datetime.")

    display(df.head())
    print()

Processing file: /content/index_files_csv/argo_bio-profile_index.csv
Converted column 'date' to datetime.
Converted column 'date_update' to datetime.


,file,date,latitude,longitude,ocean,profiler_type,institution,parameters,parameter_data_mode,date_update
0,aoml/1900722/profiles/BD1900722_001.nc,1970-01-01 05:34:21.022021624,-40.316,73.389,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
1,aoml/1900722/profiles/BD1900722_002.nc,1970-01-01 05:34:21.101064423,-40.390,73.528,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
2,aoml/1900722/profiles/BD1900722_003.nc,1970-01-01 05:34:21.111101222,-40.455,73.335,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
3,aoml/1900722/profiles/BD1900722_004.nc,1970-01-01 05:34:21.121075021,-40.134,73.080,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
4,aoml/1900722/profiles/BD1900722_005.nc,1970-01-01 05:34:21.201183300,-39.641,73.158,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230



Processing file: /content/index_files_csv/ar_index_global_prof.csv
Converted column 'date' to datetime.
Converted column 'date_update' to datetime.


,file,date,latitude,longitude,ocean,profiler_type,institution,date_update
0,aoml/13857/profiles/R13857_001.nc,1970-01-01 05:32:50.729200300,0.267,-16.032,A,845,AO,1970-01-01 05:36:21.011180520
1,aoml/13857/profiles/R13857_002.nc,1970-01-01 05:32:50.809192112,0.072,-17.659,A,845,AO,1970-01-01 05:36:21.011180521
2,aoml/13857/profiles/R13857_003.nc,1970-01-01 05:32:50.820184545,0.543,-19.622,A,845,AO,1970-01-01 05:36:21.011180521
3,aoml/13857/profiles/R13857_004.nc,1970-01-01 05:32:50.831193905,1.256,-20.521,A,845,AO,1970-01-01 05:36:21.011180521
4,aoml/13857/profiles/R13857_005.nc,1970-01-01 05:32:50.911185808,0.720,-20.768,A,845,AO,1970-01-01 05:36:21.011180521



Processing file: /content/index_files_csv/ar_index_global_traj.csv
Converted column 'date_update' to datetime.


...

,file,latitude_max,latitude_min,longitude_max,longitude_min,profiler_type,institution,parameters,date_update
0,bodc/6904189/6904189_BRtraj.nc,NaN,NaN,NaN,NaN,836,BO,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:20.110013018
1,csio/2901550/2901550_BRtraj.nc,36.813,27.921,155.622,134.477,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060644
2,csio/2901551/2901551_BRtraj.nc,29.891,23.523,149.631,139.986,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060654
3,csio/2901552/2901552_BRtraj.nc,30.627,25.121,149.524,140.608,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060700
4,csio/2901553/2901553_BRtraj.nc,36.322,27.627,162.148,140.424,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060710



Processing file: /content/index_files_csv/ar_index_global_meta.csv
Converted column 'date_update' to datetime.


,file,profiler_type,institution,date_update
0,aoml/13857/13857_meta.nc,845.0,AO,1970-01-01 05:36:21.011200014
1,aoml/13858/13858_meta.nc,845.0,AO,1970-01-01 05:36:21.011200015
2,aoml/13859/13859_meta.nc,845.0,AO,1970-01-01 05:36:21.011200025
3,aoml/15819/15819_meta.nc,845.0,AO,1970-01-01 05:36:21.011200016
4,aoml/15820/15820_meta.nc,845.0,AO,1970-01-01 05:36:21.011200018



Processing file: /content/index_files_csv/ar_index_global_tech.csv
Converted column 'date_update' to datetime.


,file,institution,date_update
0,aoml/13857/13857_tech.nc,AO,1970-01-01 05:36:50.428200335
1,aoml/13858/13858_tech.nc,AO,1970-01-01 05:36:50.428200337
2,aoml/13859/13859_tech.nc,AO,1970-01-01 05:36:21.011200025
3,aoml/15819/15819_tech.nc,AO,1970-01-01 05:36:50.428200342
4,aoml/15820/15820_tech.nc,AO,1970-01-01 05:36:50.428200346


In [5]:
import pandas as pd
import os

path = "/content/index_files_csv/"
for file_name in os.listdir(path):
    file_path = os.path.join(path, file_name)
    print(f"Processing file: {file_path}")
    df = pd.read_csv(file_path)

    # Convert columns that likely contain date information to datetime, coercing errors
    for col in df.columns:
        if 'date' in col.lower() or 'update' in col.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"Converted column '{col}' to datetime in {file_name}.")

    # Save the updated DataFrame back to the original file
    df.to_csv(file_path, index=False)
    print(f"Saved updated data to {file_name}")
    print()

Processing file: /content/index_files_csv/argo_bio-profile_index.csv
Converted column 'date' to datetime in argo_bio-profile_index.csv.
Converted column 'date_update' to datetime in argo_bio-profile_index.csv.
Saved updated data to argo_bio-profile_index.csv

Processing file: /content/index_files_csv/ar_index_global_prof.csv
Converted column 'date' to datetime in ar_index_global_prof.csv.
Converted column 'date_update' to datetime in ar_index_global_prof.csv.
Saved updated data to ar_index_global_prof.csv

Processing file: /content/index_files_csv/ar_index_global_traj.csv
Converted column 'date_update' to datetime in ar_index_global_traj.csv.
Saved updated data to ar_index_global_traj.csv

Processing file: /content/index_files_csv/argo_synthetic-profile_index.csv
Converted column 'date' to datetime in argo_synthetic-profile_index.csv.
Converted column 'date_update' to datetime in argo_synthetic-profile_index.csv.
Saved updated data to argo_synthetic-profile_index.csv

Processing file: 

In [6]:
import pandas as pd
import os # Import the os module

path = "/content/index_files_csv/"
for file_name in os.listdir(path): # Iterate over the list of filenames
    file_path = os.path.join(path, file_name) # Get the full file path
    print(f"Processing file: {file_path}")
    df = pd.read_csv(file_path)
    display(df.head())
    print()

Processing file: /content/index_files_csv/argo_bio-profile_index.csv


,file,date,latitude,longitude,ocean,profiler_type,institution,parameters,parameter_data_mode,date_update
0,aoml/1900722/profiles/BD1900722_001.nc,1970-01-01 05:34:21.022021624,-40.316,73.389,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
1,aoml/1900722/profiles/BD1900722_002.nc,1970-01-01 05:34:21.101064423,-40.390,73.528,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
2,aoml/1900722/profiles/BD1900722_003.nc,1970-01-01 05:34:21.111101222,-40.455,73.335,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
3,aoml/1900722/profiles/BD1900722_004.nc,1970-01-01 05:34:21.121075021,-40.134,73.080,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230
4,aoml/1900722/profiles/BD1900722_005.nc,1970-01-01 05:34:21.201183300,-39.641,73.158,I,846,AO,PRES TEMP_DOXY BPHASE_DOXY DOXY,RRRD,1970-01-01 05:36:40.312153230



Processing file: /content/index_files_csv/ar_index_global_prof.csv


,file,date,latitude,longitude,ocean,profiler_type,institution,date_update
0,aoml/13857/profiles/R13857_001.nc,1970-01-01 05:32:50.729200300,0.267,-16.032,A,845,AO,1970-01-01 05:36:21.011180520
1,aoml/13857/profiles/R13857_002.nc,1970-01-01 05:32:50.809192112,0.072,-17.659,A,845,AO,1970-01-01 05:36:21.011180521
2,aoml/13857/profiles/R13857_003.nc,1970-01-01 05:32:50.820184545,0.543,-19.622,A,845,AO,1970-01-01 05:36:21.011180521
3,aoml/13857/profiles/R13857_004.nc,1970-01-01 05:32:50.831193905,1.256,-20.521,A,845,AO,1970-01-01 05:36:21.011180521
4,aoml/13857/profiles/R13857_005.nc,1970-01-01 05:32:50.911185808,0.720,-20.768,A,845,AO,1970-01-01 05:36:21.011180521



Processing file: /content/index_files_csv/ar_index_global_traj.csv


...

,file,latitude_max,latitude_min,longitude_max,longitude_min,profiler_type,institution,parameters,date_update
0,bodc/6904189/6904189_BRtraj.nc,NaN,NaN,NaN,NaN,836,BO,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:20.110013018
1,csio/2901550/2901550_BRtraj.nc,36.813,27.921,155.622,134.477,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060644
2,csio/2901551/2901551_BRtraj.nc,29.891,23.523,149.631,139.986,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060654
3,csio/2901552/2901552_BRtraj.nc,30.627,25.121,149.524,140.608,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060700
4,csio/2901553/2901553_BRtraj.nc,36.322,27.627,162.148,140.424,841,HZ,PRES C1PHASE_DOXY C2PHASE_DOXY TEMP_DOXY DOXY ...,1970-01-01 05:37:21.116060710



Processing file: /content/index_files_csv/ar_index_global_meta.csv


,file,profiler_type,institution,date_update
0,aoml/13857/13857_meta.nc,845.0,AO,1970-01-01 05:36:21.011200014
1,aoml/13858/13858_meta.nc,845.0,AO,1970-01-01 05:36:21.011200015
2,aoml/13859/13859_meta.nc,845.0,AO,1970-01-01 05:36:21.011200025
3,aoml/15819/15819_meta.nc,845.0,AO,1970-01-01 05:36:21.011200016
4,aoml/15820/15820_meta.nc,845.0,AO,1970-01-01 05:36:21.011200018



Processing file: /content/index_files_csv/ar_index_global_tech.csv


,file,institution,date_update
0,aoml/13857/13857_tech.nc,AO,1970-01-01 05:36:50.428200335
1,aoml/13858/13858_tech.nc,AO,1970-01-01 05:36:50.428200337
2,aoml/13859/13859_tech.nc,AO,1970-01-01 05:36:21.011200025
3,aoml/15819/15819_tech.nc,AO,1970-01-01 05:36:50.428200342
4,aoml/15820/15820_tech.nc,AO,1970-01-01 05:36:50.428200346


In [7]:
import shutil
import os
from google.colab import files

folder_to_zip = '/content/index_files_csv'
zip_filename = 'index_files_csv.zip'

# Create a zip archive of the folder
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', folder_to_zip)

'/content/index_files_csv.zip'

In [8]:
global_prof = pd.read_csv("/content/index_files_csv/ar_index_global_prof.csv")
global_prof.head()

,file,date,latitude,longitude,ocean,profiler_type,institution,date_update
0,aoml/13857/profiles/R13857_001.nc,1970-01-01 05:32:50.729200300,0.267,-16.032,A,845,AO,1970-01-01 05:36:21.011180520
1,aoml/13857/profiles/R13857_002.nc,1970-01-01 05:32:50.809192112,0.072,-17.659,A,845,AO,1970-01-01 05:36:21.011180521
2,aoml/13857/profiles/R13857_003.nc,1970-01-01 05:32:50.820184545,0.543,-19.622,A,845,AO,1970-01-01 05:36:21.011180521
3,aoml/13857/profiles/R13857_004.nc,1970-01-01 05:32:50.831193905,1.256,-20.521,A,845,AO,1970-01-01 05:36:21.011180521
4,aoml/13857/profiles/R13857_005.nc,1970-01-01 05:32:50.911185808,0.720,-20.768,A,845,AO,1970-01-01 05:36:21.011180521


In [9]:
global_prof.shape

(3217309, 8)

In [10]:
# Assuming 'global_prof' DataFrame is already loaded and available
if 'global_prof' in locals() and isinstance(global_prof, pd.DataFrame):
    ocean_i_count = global_prof[global_prof['ocean'] == 'I'].shape[0]
    print(f"The count of rows where 'ocean' is 'I' in global_prof is: {ocean_i_count}")
else:
    print("The 'global_prof' DataFrame is not available.")

The count of rows where 'ocean' is 'I' in global_prof is: 615987


In [11]:
import os
import pandas as pd
import requests

# Folders
index_folder = "index_files_csv"
download_folder = "dac_downloads"
os.makedirs(download_folder, exist_ok=True)

# Base URL for DAC files
base_url = "https://data-argo.ifremer.fr/dac/"

# 1️⃣ Load global profile CSV
prof_csv = os.path.join(index_folder, "ar_index_global_prof.csv")
prof_df = pd.read_csv(prof_csv)

# 2️⃣ Load DAC metadata CSV (optional, for extra info)
meta_csv = os.path.join(index_folder, "ar_index_global_meta.csv")
meta_df = pd.read_csv(meta_csv)

# 3️⃣ Filter for Indian Ocean floats using 'ocean' column
indian_ocean_df = prof_df[prof_df['ocean'] == 'I']

# Optional: merge with meta to get extra info if needed
# indian_ocean_df = indian_ocean_df.merge(meta_df, on='platform_number', how='left')

print(f"Number of Indian Ocean profiles: {len(indian_ocean_df)}")




Number of Indian Ocean profiles: 615987


In [12]:
import pandas as pd
import os

# Folder with CSV index files
index_folder = "index_files_csv"
output_folder = "subset_index_csv"
os.makedirs(output_folder, exist_ok=True)

# List of all index CSVs you have
index_files = [
    "ar_index_global_prof.csv",
    "ar_index_global_meta.csv",
    "ar_index_global_tech.csv",
    "ar_index_global_traj.csv",
    "argo_bio-profile_index.csv",
    "argo_bio-traj_index.csv",
    "argo_synthetic-profile_index.csv"
]

# 1️⃣ Load global profile first to select floats
prof_df = pd.read_csv(os.path.join(index_folder, "ar_index_global_prof.csv"))

# Filter for Indian DAC (example: 'incois/')
indian_df = prof_df[prof_df['file'].str.startswith('incois/')]

# Extract platform_number from file path
# Assuming file path like 'incois/12345/profiles/R12345_001.nc'
indian_df['platform_number'] = indian_df['file'].str.split('/').str[1]

# Pick 5 unique floats
selected_floats = indian_df['platform_number'].unique()[:5]
print("Selected floats:", selected_floats)

# 2️⃣ Filter each index file for these 5 floats
for idx_file in index_files:
    idx_path = os.path.join(index_folder, idx_file)
    df = pd.read_csv(idx_path)

    # Some index files may not have 'file' column or 'platform_number', try to extract
    if 'platform_number' not in df.columns:
        if 'file' in df.columns:
            df['platform_number'] = df['file'].str.split('/').str[1]
        else:
            # Skip files that cannot be filtered by float
            print(f"Skipping {idx_file}, cannot determine platform_number")
            continue

    # Filter rows for selected floats
    subset_df = df[df['platform_number'].isin(selected_floats)]

    # Save to new CSV
    out_csv = os.path.join(output_folder, idx_file.replace(".csv", "_subset.csv"))
    subset_df.to_csv(out_csv, index=False)
    print(f"Saved subset: {out_csv}")


/tmp/ipython-input-1156385547.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  indian_df['platform_number'] = indian_df['file'].str.split('/').str[1]


Selected floats: ['1900121' '1900122' '1902669' '1902670' '1902671']
Saved subset: subset_index_csv/ar_index_global_prof_subset.csv
Saved subset: subset_index_csv/ar_index_global_meta_subset.csv
Saved subset: subset_index_csv/ar_index_global_tech_subset.csv
Saved subset: subset_index_csv/ar_index_global_traj_subset.csv
Saved subset: subset_index_csv/argo_bio-profile_index_subset.csv
Saved subset: subset_index_csv/argo_bio-traj_index_subset.csv
Saved subset: subset_index_csv/argo_synthetic-profile_index_subset.csv


In [13]:
sub1 = pd.read_csv("subset_index_csv/ar_index_global_prof_subset.csv")
sub1.shape

(370, 9)

In [14]:
path = "/content/subset_index_csv"
for file_name in os.listdir(path): # Iterate over the list of filenames
    file_path = os.path.join(path, file_name) # Get the full file path
    print(f"Processing file: {file_path}")
    df = pd.read_csv(file_path)
    print(df.shape)

Processing file: /content/subset_index_csv/ar_index_global_tech_subset.csv
(5, 4)
Processing file: /content/subset_index_csv/ar_index_global_meta_subset.csv
(5, 5)
Processing file: /content/subset_index_csv/argo_bio-profile_index_subset.csv
(0, 11)
Processing file: /content/subset_index_csv/argo_synthetic-profile_index_subset.csv
(0, 11)
Processing file: /content/subset_index_csv/argo_bio-traj_index_subset.csv
(0, 10)
Processing file: /content/subset_index_csv/ar_index_global_prof_subset.csv
(370, 9)
Processing file: /content/subset_index_csv/ar_index_global_traj_subset.csv
(2, 9)


In [15]:
"""import os
import pandas as pd
import requests

# Folder with subset index CSVs
subset_index_folder = "subset_index_csv"

# Folder to save NetCDF files
download_folder = "subset_dac_downloads"
os.makedirs(download_folder, exist_ok=True)

# Base URL for DAC
base_url = "https://data-argo.ifremer.fr/dac/"

# Function to download a file with retries
def download_file(url, local_path, retries=3):
    for attempt in range(retries):
        try:
            print(f"Downloading {url} ...")
            r = requests.get(url, stream=True, timeout=60)
            r.raise_for_status()
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
            print(f"Downloaded: {local_path}")
            return True
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
    print(f"Failed to download {url}")
    return False

# Loop through all subset index CSVs
for csv_file in os.listdir(subset_index_folder):
    if not csv_file.endswith(".csv"):
        continue

    csv_path = os.path.join(subset_index_folder, csv_file)
    df = pd.read_csv(csv_path)

    # Check 'file' column exists
    if 'file' not in df.columns:
        print(f"Skipping {csv_file}, no 'file' column")
        continue

    # Download each NetCDF file
    for dac_path in df['file']:
        file_url = base_url + dac_path
        local_file = os.path.join(download_folder, os.path.basename(dac_path))

        # Skip if already downloaded
        if os.path.exists(local_file):
            continue

        download_file(file_url, local_file)
"""

Downloaded: subset_dac_downloads/1900121_tech.nc
Downloaded: subset_dac_downloads/1900122_tech.nc
Downloaded: subset_dac_downloads/1902669_tech.nc
Downloaded: subset_dac_downloads/1902670_tech.nc
Downloaded: subset_dac_downloads/1902671_tech.nc
Downloaded: subset_dac_downloads/1900121_meta.nc
Downloaded: subset_dac_downloads/1900122_meta.nc
Downloaded: subset_dac_downloads/1902669_meta.nc
Downloaded: subset_dac_downloads/1902670_meta.nc
Downloaded: subset_dac_downloads/1902671_meta.nc
Downloaded: subset_dac_downloads/D1900121_001.nc
Downloaded: subset_dac_downloads/D1900121_002.nc
Downloaded: subset_dac_downloads/D1900121_003.nc
Downloaded: subset_dac_downloads/D1900121_004.nc
Downloaded: subset_dac_downloads/D1900121_005.nc
Downloaded: subset_dac_downloads/D1900121_006.nc
Downloaded: subset_dac_downloads/D1900121_007.nc
Downloaded: subset_dac_downloads/D1900121_008.nc
Downloaded: subset_dac_downloads/D1900121_009.nc
Downloaded: subset_dac_downloads/D1900121_010.nc
Downloaded: subset_d

In [16]:
import os

download_folder = "subset_dac_downloads"
if os.path.exists(download_folder):
    num_downloaded_files = len(os.listdir(download_folder))
    print(f"Number of files downloaded: {num_downloaded_files}")
else:
    print(f"Download folder '{download_folder}' not found.")

Number of files downloaded: 382


In [17]:
import xarray as xr
import os

# Path to one example NetCDF file
nc_file = "subset_dac_downloads/D1900121_001.nc"   # change this to one of your files

# Open with xarray
ds = xr.open_dataset(nc_file)

print("\n===== DIMENSIONS =====")
print(ds.dims)

print("\n===== VARIABLES =====")
for var in ds.data_vars:
    print(f"{var:15} → shape={ds[var].shape}, dtype={ds[var].dtype}")

print("\n===== COORDINATES =====")
for coord in ds.coords:
    print(f"{coord:15} → {ds[coord].values[:5]} ...")  # show first 5

print("\n===== ATTRIBUTES (global metadata) =====")
for attr, val in ds.attrs.items():
    print(f"{attr}: {val}")

print("\n===== SAMPLE DATA (first few rows) =====")
df = ds.to_dataframe().reset_index()
print(df.head(10))



===== DIMENSIONS =====
FrozenMappingWarningOnValuesAccess({'N_PROF': 1, 'N_PARAM': 3, 'N_LEVELS': 45, 'N_CALIB': 1, 'N_HISTORY': 9})

===== VARIABLES =====
DATA_TYPE       → shape=(), dtype=object
FORMAT_VERSION  → shape=(), dtype=object
HANDBOOK_VERSION → shape=(), dtype=object
REFERENCE_DATE_TIME → shape=(), dtype=object
DATE_CREATION   → shape=(), dtype=object
DATE_UPDATE     → shape=(), dtype=object
PLATFORM_NUMBER → shape=(1,), dtype=object
PROJECT_NAME    → shape=(1,), dtype=object
PI_NAME         → shape=(1,), dtype=object
STATION_PARAMETERS → shape=(1, 3), dtype=object
CYCLE_NUMBER    → shape=(1,), dtype=float64
DIRECTION       → shape=(1,), dtype=object
DATA_CENTRE     → shape=(1,), dtype=object
DC_REFERENCE    → shape=(1,), dtype=object
DATA_STATE_INDICATOR → shape=(1,), dtype=object
DATA_MODE       → shape=(1,), dtype=object
PLATFORM_TYPE   → shape=(1,), dtype=object
FLOAT_SERIAL_NO → shape=(1,), dtype=object
FIRMWARE_VERSION → shape=(1,), dtype=object
WMO_INST_TYPE   → sha

In [18]:
!pip install netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.8 MB/s eta 0:00:00


In [19]:
import os

folder_path = "/content/subset_dac_downloads"
if os.path.exists(folder_path):
    contents = os.listdir(folder_path)
    if contents:
        print(f"Contents of '{folder_path}':")
        for item in contents:
            print(item)
    else:
        print(f"The folder '{folder_path}' is empty.")
else:
    print(f"The folder '{folder_path}' does not exist.")

Contents of '/content/subset_dac_downloads':
D1902671_028.nc
D1900121_015.nc
R1902671_069.nc
R1902671_070.nc
D1902669_038.nc
D1902669_061.nc
D1902669_060.nc
R1902671_068.nc
D1900122_013.nc
D1900121_065.nc
D1900121_027.nc
D1900121_068.nc
D1902670_033.nc
R1902669_067.nc
D1900121_001.nc
D1902669_004.nc
D1900121_012.nc
D1902669_022.nc
D1902670_001.nc
D1902671_008.nc
D1902671_045.nc
D1900121_022.nc
D1902671_052.nc
D1900121_046.nc
D1902670_028.nc
1900121_meta.nc
D1900121_098.nc
1900121_Rtraj.nc
D1902669_036.nc
D1902670_053.nc
D1900122_041.nc
D1902669_051.nc
D1902670_018.nc
D1902671_020.nc
D1902671_006.nc
D1900121_057.nc
D1902670_051.nc
D1900122_047.nc
D1902670_058.nc
D1902670_011.nc
D1900122_006.nc
1900122_tech.nc
D1902670_031.nc
D1902669_001.nc
D1902669_018.nc
D1902669_029.nc
D1902669_054.nc
D1902671_051.nc
D1900122_004.nc
D1902671_003.nc
D1902671_019.nc
D1902671_031.nc
R1902671_063.nc
D1902670_054.nc
R1902670_062.nc
R1902671_066.nc
D1900121_023.nc
R1902669_066.nc
D1900122_028.nc
D1900122_0

In [30]:
import os
import requests
import pandas as pd
import xarray as xr

# === CONFIG ===
base_url = "https://data-argo.ifremer.fr/"     # DAC base URL
index_file = "/content/subset_index_csv/ar_index_global_prof_subset.csv"  # filtered index file
output_nc_dir = "subset_global_prof_nc"        # folder for NetCDF files
output_csv_dir = "subset_global_prof_csv"      # folder for CSVs

os.makedirs(output_nc_dir, exist_ok=True)
os.makedirs(output_csv_dir, exist_ok=True)

# === STEP 1: Load the subset index file ===
df = pd.read_csv(index_file)
print(f"✅ Subset size: {len(df)} profiles")

# === STEP 2: Download NetCDF files ===
downloaded_files = []
for i, row in df.iterrows():
    file_path = row["file"]
    url = base_url + file_path
    local_path = os.path.join(output_nc_dir, os.path.basename(file_path))

    if not os.path.exists(local_path):
        try:
            print(f"⬇️ Downloading {url}")
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(r.content)
            print(f"✅ Saved {local_path}")
        except Exception as e:
            print(f"⚠️ Failed {url}: {e}")
            continue

    downloaded_files.append(local_path)

print(f"✅ Downloaded {len(downloaded_files)} NetCDF files")

# === STEP 3: Convert each NetCDF to CSV (all variables) ===
for nc_file in downloaded_files:
    try:
        ds = xr.open_dataset(nc_file)
        df_nc = ds.to_dataframe().reset_index()  # flatten all dimensions
        csv_file = os.path.join(output_csv_dir, os.path.basename(nc_file).replace(".nc", ".csv"))
        df_nc.to_csv(csv_file, index=False)
        print(f"✅ Converted {nc_file} → {csv_file}")
        ds.close()
    except Exception as e:
        print(f"⚠️ Could not convert {nc_file}: {e}")


✅ Subset size: 370 profiles
⬇️ Downloading https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_001.nc
⚠️ Failed https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_001.nc: 404 Client Error: Not Found for url: https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_001.nc
⬇️ Downloading https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_002.nc
⚠️ Failed https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_002.nc: 404 Client Error: Not Found for url: https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_002.nc
⬇️ Downloading https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_003.nc
⚠️ Failed https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_003.nc: 404 Client Error: Not Found for url: https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_003.nc
⬇️ Downloading https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_004.nc
⚠️ Failed https://data-argo.ifremer.fr/incois/1900121/profiles/D1900121_004.nc

KeyboardInterrupt: 

In [27]:
import requests
import os

url = "https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D1900121_001.nc"
local_path = os.path.basename(url)

try:
    print(f"Downloading {url} ...")
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(local_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)
    print(f"Downloaded: {local_path}")
except Exception as e:
    print(f"Failed to download {url}: {e}")

Downloaded: D1900121_001.nc


In [32]:
import os
import requests
from bs4 import BeautifulSoup

# === CONFIG ===
base_url = "https://data-argo.ifremer.fr/dac/incois/"
float_ids = ['1900121','1900122','1902669','1902670','1902671']  # 5 example floats
output_folder = "incois_floats_nc"
os.makedirs(output_folder, exist_ok=True)

for float_id in float_ids:
    folder_url = f"{base_url}{float_id}/profiles/"
    print(f"🌊 Processing float {float_id}: {folder_url}")

    try:
        # Get HTML page
        r = requests.get(folder_url, timeout=60)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        # Extract .nc links
        nc_files = [a['href'] for a in soup.find_all('a') if a['href'].endswith(".nc")]
        print(f"Found {len(nc_files)} .nc files")

        # Create folder for this float
        float_folder = os.path.join(output_folder, float_id)
        os.makedirs(float_folder, exist_ok=True)

        # Download each .nc
        for nc_file in nc_files:
            nc_url = folder_url + nc_file
            local_path = os.path.join(float_folder, nc_file)

            if not os.path.exists(local_path):
                try:
                    print(f"⬇️ Downloading {nc_url}")
                    r_file = requests.get(nc_url, timeout=60)
                    r_file.raise_for_status()
                    with open(local_path, "wb") as f:
                        f.write(r_file.content)
                    print(f"✅ Saved {local_path}")
                except Exception as e:
                    print(f"⚠️ Failed {nc_url}: {e}")

    except Exception as e:
        print(f"⚠️ Could not access folder {folder_url}: {e}")


🌊 Processing float 1900121: https://data-argo.ifremer.fr/dac/incois/1900121/profiles/
Found 99 .nc files
⬇️ Downloading https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D1900121_001.nc
✅ Saved incois_floats_nc/1900121/D1900121_001.nc
⬇️ Downloading https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D1900121_002.nc
✅ Saved incois_floats_nc/1900121/D1900121_002.nc
⬇️ Downloading https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D1900121_003.nc
✅ Saved incois_floats_nc/1900121/D1900121_003.nc
⬇️ Downloading https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D1900121_004.nc
✅ Saved incois_floats_nc/1900121/D1900121_004.nc
⬇️ Downloading https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D1900121_005.nc
✅ Saved incois_floats_nc/1900121/D1900121_005.nc
⬇️ Downloading https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D1900121_006.nc
✅ Saved incois_floats_nc/1900121/D1900121_006.nc
⬇️ Downloading https://data-argo.ifremer.fr/dac/incois/1900121/profiles/D

In [195]:
import os
import pandas as pd
import xarray as xr
from tqdm import tqdm  # nice progress bar

# === CONFIG ===
floats_folder = "incois_floats_nc"  # folder containing all float subfolders
output_csv = "incois_5floats_combined.csv"

all_rows = []

# Traverse all float subfolders
for float_id in os.listdir(floats_folder):
    float_path = os.path.join(floats_folder, float_id)
    if not os.path.isdir(float_path):
        continue

    nc_files = [f for f in os.listdir(float_path) if f.endswith(".nc")]
    print(f"Processing float {float_id}: {len(nc_files)} files")

    for nc_file in tqdm(nc_files, desc=f"Float {float_id}"):
        nc_path = os.path.join(float_path, nc_file)
        try:
            ds = xr.open_dataset(nc_path)
            df_nc = ds.to_dataframe().reset_index()  # flatten all dimensions
            df_nc["float_id"] = float_id            # keep track of float
            df_nc["file_name"] = nc_file            # keep track of file
            all_rows.append(df_nc)
            ds.close()
        except Exception as e:
            print(f"⚠️ Could not read {nc_path}: {e}")

# Combine all DataFrames
if all_rows:
    combined_df = pd.concat(all_rows, ignore_index=True)
    combined_df.to_csv(output_csv, index=False)
    print(f"✅ Combined CSV saved: {output_csv}")
else:
    print("⚠️ No data to save")


Processing float 1900121: 99 files


Float 1900121: 100%|██████████| 99/99 [00:05<00:00, 19.58it/s]


Processing float 1900122: 52 files


Float 1900122: 100%|██████████| 52/52 [00:01<00:00, 27.77it/s]


Processing float 1902670: 73 files


...

Processing float 1902669: 74 files


Float 1902669: 100%|██████████| 74/74 [00:02<00:00, 26.79it/s]


Processing float 1902671: 72 files


Float 1902671: 100%|██████████| 72/72 [00:02<00:00, 27.94it/s]


✅ Combined CSV saved: incois_5floats_combined.csv


In [196]:
import xarray as xr
import os

# Path to one example NetCDF file
nc_file = "/content/D1900121_001.nc"   # change this to one of your files

# Open with xarray
ds = xr.open_dataset(nc_file)

print("\n===== DIMENSIONS =====")
print(ds.dims)

print("\n===== VARIABLES =====")
for var in ds.data_vars:
    print(f"{var:15} → shape={ds[var].shape}, dtype={ds[var].dtype}")

print("\n===== COORDINATES =====")
for coord in ds.coords:
    print(f"{coord:15} → {ds[coord].values[:5]} ...")  # show first 5

print("\n===== ATTRIBUTES (global metadata) =====")
for attr, val in ds.attrs.items():
    print(f"{attr}: {val}")

print("\n===== SAMPLE DATA (first few rows) =====")
df_sample = ds.to_dataframe().reset_index()
print(df_sample.head(10))
ds.close() # Close the dataset after use


===== DIMENSIONS =====
FrozenMappingWarningOnValuesAccess({'N_PROF': 1, 'N_PARAM': 3, 'N_LEVELS': 45, 'N_CALIB': 1, 'N_HISTORY': 9})

===== VARIABLES =====
DATA_TYPE       → shape=(), dtype=object
FORMAT_VERSION  → shape=(), dtype=object
HANDBOOK_VERSION → shape=(), dtype=object
REFERENCE_DATE_TIME → shape=(), dtype=object
DATE_CREATION   → shape=(), dtype=object
DATE_UPDATE     → shape=(), dtype=object
PLATFORM_NUMBER → shape=(1,), dtype=object
PROJECT_NAME    → shape=(1,), dtype=object
PI_NAME         → shape=(1,), dtype=object
STATION_PARAMETERS → shape=(1, 3), dtype=object
CYCLE_NUMBER    → shape=(1,), dtype=float64
DIRECTION       → shape=(1,), dtype=object
DATA_CENTRE     → shape=(1,), dtype=object
DC_REFERENCE    → shape=(1,), dtype=object
DATA_STATE_INDICATOR → shape=(1,), dtype=object
DATA_MODE       → shape=(1,), dtype=object
PLATFORM_TYPE   → shape=(1,), dtype=object
FLOAT_SERIAL_NO → shape=(1,), dtype=object
FIRMWARE_VERSION → shape=(1,), dtype=object
WMO_INST_TYPE   → sha

In [197]:
df_sample.shape

(1215, 69)

In [198]:
df = pd.read_csv("incois_5floats_combined.csv")
df.head()

,N_PROF,N_PARAM,N_LEVELS,N_CALIB,N_HISTORY,DATA_TYPE,FORMAT_VERSION,HANDBOOK_VERSION,REFERENCE_DATE_TIME,DATE_CREATION,...,HISTORY_REFERENCE,HISTORY_DATE,HISTORY_ACTION,HISTORY_PARAMETER,HISTORY_START_PRES,HISTORY_STOP_PRES,HISTORY_PREVIOUS_VALUE,HISTORY_QCTEST,float_id,file_name
0,0,0,0,0,0,b'Argo profile ',b'3.1 ',b' 1.2',b'19500101000000',b'20030331222143',...,b' ...,b'20090218152619',b' IP',b' ',NaN,NaN,NaN,b' ',1900121,D1900121_015.nc
1,0,0,0,0,1,b'Argo profile ',b'3.1 ',b' 1.2',b'19500101000000',b'20030331222143',...,b' ...,b'20090218152619',b' IP',b' ',NaN,NaN,NaN,b' ',1900121,D1900121_015.nc
2,0,0,0,0,2,b'Argo profile ',b'3.1 ',b' 1.2',b'19500101000000',b'20030331222143',...,b' ...,b'20090218152619',b' IP',b' ',NaN,NaN,NaN,b' ',1900121,D1900121_015.nc
3,0,0,0,0,3,b'Argo profile ',b'3.1 ',b' 1.2',b'19500101000000',b'20030331222143',...,b' ...,b'20090218152619',b' IP',b' ',NaN,NaN,NaN,b' ',1900121,D1900121_015.nc
4,0,0,0,0,4,b'Argo profile ',b'3.1 ',b' 1.2',b'19500101000000',b'20030331222143',...,b' ...,b'20090218152619',b'QCP$',b' ',NaN,NaN,NaN,b'D7B7E ',1900121,D1900121_015.nc


In [199]:
df.columns

Index(['N_PROF', 'N_PARAM', 'N_LEVELS', 'N_CALIB', 'N_HISTORY', 'DATA_TYPE',
       'FORMAT_VERSION', 'HANDBOOK_VERSION', 'REFERENCE_DATE_TIME',
       'DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'PROJECT_NAME',
       'PI_NAME', 'STATION_PARAMETERS', 'CYCLE_NUMBER', 'DIRECTION',
       'DATA_CENTRE', 'DC_REFERENCE', 'DATA_STATE_INDICATOR', 'DATA_MODE',
       'PLATFORM_TYPE', 'FLOAT_SERIAL_NO', 'FIRMWARE_VERSION', 'WMO_INST_TYPE',
       'JULD', 'JULD_QC', 'JULD_LOCATION', 'LATITUDE', 'LONGITUDE',
       'POSITION_QC', 'POSITIONING_SYSTEM', 'PROFILE_PRES_QC',
       'PROFILE_TEMP_QC', 'PROFILE_PSAL_QC', 'VERTICAL_SAMPLING_SCHEME',
       'CONFIG_MISSION_NUMBER', 'PRES', 'PRES_QC', 'PRES_ADJUSTED',
       'PRES_ADJUSTED_QC', 'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED',
       'TEMP_ADJUSTED_QC', 'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED',
       'PSAL_ADJUSTED_QC', 'PRES_ADJUSTED_ERROR', 'TEMP_ADJUSTED_ERROR',
       'PSAL_ADJUSTED_ERROR', 'PARAMETER', 'SCIENTIFIC_CALIB_EQUATION',
       'SCIENT

In [59]:
import pandas as pd

# Assuming df is the DataFrame loaded from the combined CSV
# Select object columns
object_cols = df.select_dtypes(include='object').columns

# Clean strings in object columns
for col in object_cols:
    df[col] = df[col].astype(str).str.replace("b'", "", regex=False).str.replace("'", "", regex=False).str.strip()

print("Object columns cleaned.")
display(df.head())

Object columns cleaned.


,float_id,file_name,PLATFORM_NUMBER,CYCLE_NUMBER,JULD,LATITUDE,LONGITUDE,PRES,TEMP,PSAL,...,TEMP_QC,PSAL_QC,PRES_ADJUSTED,TEMP_ADJUSTED,PSAL_ADJUSTED,PRES_ADJUSTED_QC,TEMP_ADJUSTED_QC,PSAL_ADJUSTED_QC,DATA_MODE,PLATFORM_TYPE
0,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
1,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
2,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
3,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
4,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX


In [60]:
df.shape

(627804, 21)

In [61]:
# Remove duplicate rows from the DataFrame
df_no_duplicates = df.drop_duplicates()

print(f"Original number of rows: {len(df)}")
print(f"Number of rows after removing duplicates: {len(df_no_duplicates)}")

# You can now use df_no_duplicates for further analysis
display(df_no_duplicates.head())

Original number of rows: 627804
Number of rows after removing duplicates: 28473


,float_id,file_name,PLATFORM_NUMBER,CYCLE_NUMBER,JULD,LATITUDE,LONGITUDE,PRES,TEMP,PSAL,...,TEMP_QC,PSAL_QC,PRES_ADJUSTED,TEMP_ADJUSTED,PSAL_ADJUSTED,PRES_ADJUSTED_QC,TEMP_ADJUSTED_QC,PSAL_ADJUSTED_QC,DATA_MODE,PLATFORM_TYPE
0,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.860,...,1,1,4.300000,30.137,34.861885,1,1,1,D,APEX
9,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,9.3,30.061,34.863,...,1,1,5.900000,30.061,34.864216,1,1,1,D,APEX
18,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,14.1,30.018,34.866,...,1,1,10.700001,30.018,34.867400,1,1,1,D,APEX
27,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,19.2,30.005,34.869,...,1,1,15.800001,30.005,34.875480,1,1,1,D,APEX
36,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,24.7,28.931,34.908,...,1,1,21.300001,28.931,34.911694,1,1,1,D,APEX


In [62]:
df_no_duplicates.shape

(28473, 21)

In [63]:
df.columns

Index(['float_id', 'file_name', 'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'JULD',
       'LATITUDE', 'LONGITUDE', 'PRES', 'TEMP', 'PSAL', 'PRES_QC', 'TEMP_QC',
       'PSAL_QC', 'PRES_ADJUSTED', 'TEMP_ADJUSTED', 'PSAL_ADJUSTED',
       'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC', 'DATA_MODE',
       'PLATFORM_TYPE'],
      dtype='object')

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 627804 entries, 0 to 627803
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   float_id          627804 non-null  int64  
 1   file_name         627804 non-null  object 
 2   PLATFORM_NUMBER   627804 non-null  object 
 3   CYCLE_NUMBER      627804 non-null  float64
 4   JULD              627804 non-null  object 
 5   LATITUDE          627804 non-null  float64
 6   LONGITUDE         627804 non-null  float64
 7   PRES              627669 non-null  float64
 8   TEMP              627669 non-null  float64
 9   PSAL              627669 non-null  float64
 10  PRES_QC           627804 non-null  object 
 11  TEMP_QC           627804 non-null  object 
 12  PSAL_QC           627804 non-null  object 
 13  PRES_ADJUSTED     561828 non-null  float64
 14  TEMP_ADJUSTED     561420 non-null  float64
 15  PSAL_ADJUSTED     561450 non-null  float64
 16  PRES_ADJUSTED_QC  62

In [65]:
core_columns = [
    'float_id', 'file_name', 'PLATFORM_NUMBER', 'CYCLE_NUMBER',
    'JULD', 'LATITUDE', 'LONGITUDE', 'PRES', 'TEMP', 'PSAL',
    'PRES_QC', 'TEMP_QC', 'PSAL_QC',
    'PRES_ADJUSTED', 'TEMP_ADJUSTED', 'PSAL_ADJUSTED',
    'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC',
    'DATA_MODE', 'PLATFORM_TYPE'
]

existing_columns = [col for col in core_columns if col in df.columns]
df_core = df[existing_columns]
df_core.head()

,float_id,file_name,PLATFORM_NUMBER,CYCLE_NUMBER,JULD,LATITUDE,LONGITUDE,PRES,TEMP,PSAL,...,TEMP_QC,PSAL_QC,PRES_ADJUSTED,TEMP_ADJUSTED,PSAL_ADJUSTED,PRES_ADJUSTED_QC,TEMP_ADJUSTED_QC,PSAL_ADJUSTED_QC,DATA_MODE,PLATFORM_TYPE
0,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
1,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
2,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
3,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX
4,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.86,...,1,1,4.3,30.137,34.861885,1,1,1,D,APEX


In [66]:


# === STEP 3: Save filtered CSV ===
df_core.to_csv(output_csv, index=False)
print(f"✅ Core CSV saved: {output_csv}")
print(f"Columns retained: {existing_columns}")
print(f"Rows: {len(df_core)}")


✅ Core CSV saved: incois_5floats_combined.csv
Columns retained: ['float_id', 'file_name', 'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'JULD', 'LATITUDE', 'LONGITUDE', 'PRES', 'TEMP', 'PSAL', 'PRES_QC', 'TEMP_QC', 'PSAL_QC', 'PRES_ADJUSTED', 'TEMP_ADJUSTED', 'PSAL_ADJUSTED', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC', 'DATA_MODE', 'PLATFORM_TYPE']
Rows: 627804


In [69]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd
import xarray as xr
from tqdm import tqdm

# ================= CONFIG =================
base_url = "https://data-argo.ifremer.fr/dac/incois/"
float_ids = ['1900121','1900122','1902669','1902670','1902671']  # 5 floats
output_nc_dir = "incois_floats_nc"      # Folder to store NetCDFs
output_csv = "incois_5floats_combined.csv"  # Final CSV
core_columns = [
    'float_id', 'file_name', 'PLATFORM_NUMBER', 'CYCLE_NUMBER',
    'JULD', 'LATITUDE', 'LONGITUDE', 'PRES', 'TEMP', 'PSAL',
    'PRES_QC', 'TEMP_QC', 'PSAL_QC',
    'PRES_ADJUSTED', 'TEMP_ADJUSTED', 'PSAL_ADJUSTED',
    'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC',
    'DATA_MODE', 'PLATFORM_TYPE'
]

os.makedirs(output_nc_dir, exist_ok=True)

# ================= FUNCTIONS =================
def download_float_nc(float_id):
    """Download all NetCDF files for a given float"""
    folder_url = f"{base_url}{float_id}/profiles/"
    float_folder = os.path.join(output_nc_dir, float_id)
    os.makedirs(float_folder, exist_ok=True)

    try:
        r = requests.get(folder_url, timeout=60)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        nc_files = [a['href'] for a in soup.find_all('a') if a['href'].endswith(".nc")]
        print(f"🌊 Float {float_id}: Found {len(nc_files)} .nc files")

        for nc_file in tqdm(nc_files, desc=f"Downloading {float_id}"):
            local_path = os.path.join(float_folder, nc_file)
            if not os.path.exists(local_path):
                try:
                    r_file = requests.get(folder_url + nc_file, timeout=60)
                    r_file.raise_for_status()
                    with open(local_path, "wb") as f:
                        f.write(r_file.content)
                except Exception as e:
                    print(f"⚠️ Failed {nc_file}: {e}")
        return float_folder
    except Exception as e:
        print(f"⚠️ Could not access folder {folder_url}: {e}")
        return None

def nc_to_df(nc_path, float_id):
    """Convert a single NetCDF file to a flattened DataFrame"""
    try:
        ds = xr.open_dataset(nc_path)
        df_nc = ds.to_dataframe().reset_index()
        df_nc["float_id"] = float_id
        df_nc["file_name"] = os.path.basename(nc_path)
        ds.close()
        return df_nc
    except Exception as e:
        print(f"⚠️ Could not read {nc_path}: {e}")
        return pd.DataFrame()

def clean_object_columns(df):
    """Clean object/string columns"""
    object_cols = df.select_dtypes(include='object').columns
    for col in object_cols:
        df[col] = df[col].astype(str).str.replace("b'", "", regex=False).str.replace("'", "", regex=False).str.strip()
    return df

# ================= MAIN PROCESS =================
all_dfs = []

for float_id in float_ids:
    float_folder = download_float_nc(float_id)
    if not float_folder:
        continue

    nc_files = [os.path.join(float_folder, f) for f in os.listdir(float_folder) if f.endswith(".nc")]
    for nc_file in tqdm(nc_files, desc=f"Processing {float_id}"):
        df_nc = nc_to_df(nc_file, float_id)
        if not df_nc.empty:
            all_dfs.append(df_nc)

if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = clean_object_columns(combined_df)

    # Keep only core columns
    existing_cols = [col for col in core_columns if col in combined_df.columns]
    combined_df = combined_df[existing_cols]

    combined_df.to_csv(output_csv, index=False)
    print(f"✅ Final CSV saved: {output_csv}")
    print(f"Columns retained: {existing_cols}")
    print(f"Total rows: {len(combined_df)}")
else:
    print("⚠️ No data to save")


🌊 Float 1900121: Found 99 .nc files


Processing 1900121: 100%|██████████| 99/99 [00:04<00:00, 20.26it/s]


🌊 Float 1900122: Found 52 .nc files


Processing 1900122: 100%|██████████| 52/52 [00:01<00:00, 29.36it/s]


🌊 Float 1902669: Found 74 .nc files


...

🌊 Float 1902670: Found 73 .nc files


Processing 1902670: 100%|██████████| 73/73 [00:02<00:00, 25.83it/s]


🌊 Float 1902671: Found 72 .nc files


Processing 1902671: 100%|██████████| 72/72 [00:03<00:00, 19.35it/s]


✅ Final CSV saved: incois_5floats_combined.csv
Columns retained: ['float_id', 'file_name', 'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'JULD', 'LATITUDE', 'LONGITUDE', 'PRES', 'TEMP', 'PSAL', 'PRES_QC', 'TEMP_QC', 'PSAL_QC', 'PRES_ADJUSTED', 'TEMP_ADJUSTED', 'PSAL_ADJUSTED', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC', 'DATA_MODE', 'PLATFORM_TYPE']
Total rows: 627804


In [158]:
# Check for missing values in each column
missing_values = df.isnull().sum()

# Display columns with missing values
print("Missing values per column:")
print(missing_values[missing_values > 0])

Missing values per column:
PRES                   1
TEMP                   1
PSAL                   1
PRES_ADJUSTED       3612
TEMP_ADJUSTED       3625
PSAL_ADJUSTED       3624
PRES_ADJUSTED_QC    3543
TEMP_ADJUSTED_QC    3543
PSAL_ADJUSTED_QC    3543
dtype: int64


In [157]:
df = pd.read_csv("incois_5floats_combined.csv")
# Calculate the percentage of missing values for each column
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Define a threshold for dropping columns (e.g., 50% missing values)
threshold = 50

# Get the list of columns to drop
cols_to_drop = missing_percentage[missing_percentage > threshold].index.tolist()

# Drop the columns from the DataFrame
df = df.drop(columns=cols_to_drop)

print(f"Columns dropped (with > {threshold}% missing values): {cols_to_drop}")
print(f"Shape of DataFrame after dropping columns: {df.shape}")

df = df.drop_duplicates()
# Display the head of the new DataFrame
display(df.head())

Columns dropped (with > 50% missing values): []
Shape of DataFrame after dropping columns: (627804, 21)


,float_id,file_name,PLATFORM_NUMBER,CYCLE_NUMBER,JULD,LATITUDE,LONGITUDE,PRES,TEMP,PSAL,...,TEMP_QC,PSAL_QC,PRES_ADJUSTED,TEMP_ADJUSTED,PSAL_ADJUSTED,PRES_ADJUSTED_QC,TEMP_ADJUSTED_QC,PSAL_ADJUSTED_QC,DATA_MODE,PLATFORM_TYPE
0,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,7.7,30.137,34.860,...,1,1,4.300000,30.137,34.861885,1.0,1.0,1.0,D,APEX
9,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,9.3,30.061,34.863,...,1,1,5.900000,30.061,34.864216,1.0,1.0,1.0,D,APEX
18,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,14.1,30.018,34.866,...,1,1,10.700001,30.018,34.867400,1.0,1.0,1.0,D,APEX
27,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,19.2,30.005,34.869,...,1,1,15.800001,30.005,34.875480,1.0,1.0,1.0,D,APEX
36,1900121,D1900121_015.nc,1900121,15.0,2003-03-31 09:16:27.999992064,-10.114,54.538,24.7,28.931,34.908,...,1,1,21.300001,28.931,34.911694,1.0,1.0,1.0,D,APEX


In [160]:
df = df.drop_duplicates()
print(df.shape)
df.to_csv(output_csv, index=False)
print(f"✅ Core CSV saved: {output_csv}")
print(f"Columns retained: {existing_columns}")
print(f"Rows: {len(df)}")

(28155, 21)
✅ Core CSV saved: incois_csv_all_indexes/incois_profiles_combined.csv
Columns retained: ['float_id', 'file_name', 'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'JULD', 'LATITUDE', 'LONGITUDE', 'PRES', 'TEMP', 'PSAL', 'PRES_QC', 'TEMP_QC', 'PSAL_QC', 'PRES_ADJUSTED', 'TEMP_ADJUSTED', 'PSAL_ADJUSTED', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC', 'DATA_MODE', 'PLATFORM_TYPE']
Rows: 28155


In [164]:
import os
import pandas as pd
import xarray as xr
import requests
from tqdm import tqdm

# === CONFIG ===
BASE_URL = "https://data-argo.ifremer.fr/dac/"  # DAC base URL including /dac/
INDEX_FILE = "/content/subset_index_csv/ar_index_global_meta_subset.csv"  # subset index CSV for 5 floats
OUTPUT_FOLDER = "global_meta_nc"                # folder to save downloaded NetCDF files
OUTPUT_CSV = "global_meta_5_floats_full.csv"   # final combined CSV

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# === FUNCTION: Download NetCDF ===
def download_nc(file_path, save_dir):
    url = BASE_URL + file_path
    local_path = os.path.join(save_dir, os.path.basename(file_path))

    if os.path.exists(local_path):
        return local_path

    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(r.content)
        return local_path
    except Exception as e:
        print(f"⚠️ Failed {file_path}: {e}")
        return None

# === FUNCTION: Convert NetCDF to DataFrame (keep all columns) ===
def nc_to_df(nc_file):
    try:
        ds = xr.open_dataset(nc_file)
        df = ds.to_dataframe().reset_index()
        ds.close()
        df["file_name"] = os.path.basename(nc_file)  # Keep file name for reference
        return df
    except Exception as e:
        print(f"⚠️ Could not read {nc_file}: {e}")
        return pd.DataFrame()  # empty DataFrame if read fails

# === STEP 1: Load index CSV ===
df_index = pd.read_csv(INDEX_FILE)
print(f"✅ Index loaded: {len(df_index)} files to process")

# === STEP 2: Loop through each file, download and convert ===
all_dfs = []

for i, row in tqdm(df_index.iterrows(), total=len(df_index)):
    nc_file = download_nc(row["file"], OUTPUT_FOLDER)
    if nc_file:
        df = nc_to_df(nc_file)
        if not df.empty:
            all_dfs.append(df)

# === STEP 3: Optional cleaning functions ===
def clean_object_columns(df):
    """Clean object/string columns"""
    object_cols = df.select_dtypes(include='object').columns
    for col in object_cols:
        df[col] = df[col].astype(str).str.replace("b'", "", regex=False)\
                                    .str.replace("'", "", regex=False)\
                                    .str.strip()
    return df

def format_date_columns(df):
    """Convert date columns to datetime objects"""
    for col in df.columns:
        if 'date' in col.lower() or 'juld' in col.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"Converted column '{col}' to datetime.")
    return df

def drop_cols_with_high_nulls(df, threshold=50):
    """
    Drops columns from a DataFrame that have a percentage of missing values
    above a specified threshold.

    Args:
        df: The pandas DataFrame to process.
        threshold: The percentage of missing values above which a column will be dropped (default is 50).

    Returns:
        The DataFrame with columns dropped.
    """
    # Calculate the percentage of missing values for each column
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    # Get the list of columns to drop
    cols_to_drop = missing_percentage[missing_percentage > threshold].index.tolist()

    # Drop the columns from the DataFrame
    df_dropped = df.drop(columns=cols_to_drop)

    print(f"Columns dropped (with > {threshold}% missing values): {cols_to_drop}")
    print(f"Shape of DataFrame after dropping columns: {df_dropped.shape}")

    return df_dropped

# === STEP 4: Combine all into single CSV ===
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df = clean_object_columns(final_df)
    final_df = final_df.drop_duplicates()
    final_df = format_date_columns(final_df)
    final_df = drop_cols_with_high_nulls(final_df, threshold=60)
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Final combined CSV saved: {OUTPUT_CSV}")
else:
    print("⚠️ No data was converted!")


✅ Index loaded: 5 files to process


100%|██████████| 5/5 [00:00<00:00, 31.30it/s]


Converted column 'DATE_CREATION' to datetime.
Converted column 'DATE_UPDATE' to datetime.
Converted column 'LAUNCH_DATE' to datetime.
Converted column 'START_DATE' to datetime.
Converted column 'START_DATE_QC' to datetime.
Converted column 'STARTUP_DATE' to datetime.
Converted column 'STARTUP_DATE_QC' to datetime.
Converted column 'END_MISSION_DATE' to datetime.
Columns dropped (with > 60% missing values): ['START_DATE_QC', 'STARTUP_DATE', 'STARTUP_DATE_QC', 'END_MISSION_DATE']
Shape of DataFrame after dropping columns: (7956, 69)
✅ Final combined CSV saved: global_meta_5_floats_full.csv


In [165]:
import pandas as pd

# Load the full CSV
full_csv = "global_meta_5_floats_full.csv"
df = pd.read_csv(full_csv)
df = df.drop_duplicates()

# Define the important columns you want to keep
important_cols = [
    "PLATFORM_NUMBER", "FLOAT_SERIAL_NO", "PLATFORM_TYPE", "PLATFORM_FAMILY",
    "PLATFORM_MAKER", "DATA_CENTRE", "PROJECT_NAME", "PI_NAME",
    "DEPLOYMENT_PLATFORM", "LAUNCH_DATE", "LAUNCH_LATITUDE", "LAUNCH_LONGITUDE",
    "END_MISSION_DATE", "SENSOR", "PARAMETER", "file_name"
]


In [166]:
# Check for missing values in each column
missing_values = df.isnull().sum()

# Display columns with missing values
print("Missing values per column:")
print(missing_values[missing_values > 0])

Missing values per column:
TRANS_SYSTEM_ID                         5292
TRANS_FREQUENCY                         5292
FIRMWARE_VERSION                         900
MANUAL_VERSION                           900
ANOMALY                                 7956
CONTROLLER_BOARD_TYPE_PRIMARY           6192
CONTROLLER_BOARD_TYPE_SECONDARY         7956
CONTROLLER_BOARD_SERIAL_NO_SECONDARY    7956
SPECIAL_FEATURES                        7956
CUSTOMISATION                           7956
DEPLOYMENT_CRUISE_ID                    7956
DEPLOYMENT_REFERENCE_STATION_ID         7956
END_MISSION_STATUS                      5292
CONFIG_MISSION_COMMENT                  7956
PREDEPLOYMENT_CALIB_COMMENT             7956
dtype: int64


In [167]:

# Keep only the columns that exist in the dataframe
cols_to_keep = [c for c in important_cols if c in df.columns]
df_imp = df[cols_to_keep]

# Save filtered CSV
df_imp = df_imp.drop_duplicates()
df_imp.to_csv("global_meta_5_floats.csv", index=False)
print(f"✅ Important columns CSV saved: global_meta_5_floats.csv")


✅ Important columns CSV saved: global_meta_5_floats.csv


In [168]:
df = pd.read_csv("global_meta_5_floats.csv")
df.head()

,PLATFORM_NUMBER,FLOAT_SERIAL_NO,PLATFORM_TYPE,PLATFORM_FAMILY,PLATFORM_MAKER,DATA_CENTRE,PROJECT_NAME,PI_NAME,DEPLOYMENT_PLATFORM,LAUNCH_DATE,LAUNCH_LATITUDE,LAUNCH_LONGITUDE,SENSOR,PARAMETER,file_name
0,1900121,763,APEX,FLOAT,WRC,IN,Argo INDIA,M Ravichandran,orv sagar kanya,2002-11-02 02:00:00,-10.0,56.0,CTD_TEMP,TEMP,1900121_meta.nc
1,1900121,763,APEX,FLOAT,WRC,IN,Argo INDIA,M Ravichandran,orv sagar kanya,2002-11-02 02:00:00,-10.0,56.0,CTD_TEMP,PSAL,1900121_meta.nc
2,1900121,763,APEX,FLOAT,WRC,IN,Argo INDIA,M Ravichandran,orv sagar kanya,2002-11-02 02:00:00,-10.0,56.0,CTD_TEMP,PRES,1900121_meta.nc
3,1900121,763,APEX,FLOAT,WRC,IN,Argo INDIA,M Ravichandran,orv sagar kanya,2002-11-02 02:00:00,-10.0,56.0,CTD_CNDC,TEMP,1900121_meta.nc
4,1900121,763,APEX,FLOAT,WRC,IN,Argo INDIA,M Ravichandran,orv sagar kanya,2002-11-02 02:00:00,-10.0,56.0,CTD_CNDC,PSAL,1900121_meta.nc


In [169]:
df.shape

(45, 15)

In [170]:
import os
import pandas as pd
import xarray as xr
import requests
from tqdm import tqdm

# === CONFIG ===
BASE_URL = "https://data-argo.ifremer.fr/dac/"  # DAC base URL including /dac/
INDEX_FILE = "/content/subset_index_csv/ar_index_global_tech_subset.csv"  # subset index CSV for 5 floats
OUTPUT_FOLDER = "global_tech_nc"                # folder to save downloaded NetCDF files
OUTPUT_CSV = "global_tech_5_floats_full.csv"   # final combined CSV

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# === FUNCTION: Download NetCDF ===
def download_nc(file_path, save_dir):
    url = BASE_URL + file_path
    local_path = os.path.join(save_dir, os.path.basename(file_path))

    if os.path.exists(local_path):
        return local_path

    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(r.content)
        return local_path
    except Exception as e:
        print(f"⚠️ Failed {file_path}: {e}")
        return None

# === FUNCTION: Convert NetCDF to DataFrame (keep all columns) ===
def nc_to_df(nc_file):
    try:
        ds = xr.open_dataset(nc_file)
        df = ds.to_dataframe().reset_index()
        ds.close()
        df["file_name"] = os.path.basename(nc_file)  # Keep file name for reference
        return df
    except Exception as e:
        print(f"⚠️ Could not read {nc_file}: {e}")
        return pd.DataFrame()  # empty DataFrame if read fails

# === STEP 1: Load index CSV ===
df_index = pd.read_csv(INDEX_FILE)
print(f"✅ Index loaded: {len(df_index)} files to process")

# === STEP 2: Loop through each file, download and convert ===
all_dfs = []

for i, row in tqdm(df_index.iterrows(), total=len(df_index)):
    nc_file = download_nc(row["file"], OUTPUT_FOLDER)
    if nc_file:
        df = nc_to_df(nc_file)
        if not df.empty:
            all_dfs.append(df)

# === STEP 3: Optional cleaning functions ===
def clean_object_columns(df):
    """Clean object/string columns"""
    object_cols = df.select_dtypes(include='object').columns
    for col in object_cols:
        df[col] = df[col].astype(str).str.replace("b'", "", regex=False)\
                                    .str.replace("'", "", regex=False)\
                                    .str.strip()
    return df

def format_date_columns(df):
    """Convert date columns to datetime objects"""
    for col in df.columns:
        if 'date' in col.lower() or 'juld' in col.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"Converted column '{col}' to datetime.")
    return df

def drop_cols_with_high_nulls(df, threshold=50):
    """
    Drops columns from a DataFrame that have a percentage of missing values
    above a specified threshold.

    Args:
        df: The pandas DataFrame to process.
        threshold: The percentage of missing values above which a column will be dropped (default is 50).

    Returns:
        The DataFrame with columns dropped.
    """
    # Calculate the percentage of missing values for each column
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    # Get the list of columns to drop
    cols_to_drop = missing_percentage[missing_percentage > threshold].index.tolist()

    # Drop the columns from the DataFrame
    df_dropped = df.drop(columns=cols_to_drop)

    print(f"Columns dropped (with > {threshold}% missing values): {cols_to_drop}")
    print(f"Shape of DataFrame after dropping columns: {df_dropped.shape}")

    return df_dropped

# === STEP 4: Combine all into single CSV ===
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df = clean_object_columns(final_df)
    final_df = final_df.drop_duplicates()
    final_df = format_date_columns(final_df)
    final_df = drop_cols_with_high_nulls(final_df, threshold=60)
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Final combined CSV saved: {OUTPUT_CSV}")
else:
    print("⚠️ No data was converted!")


✅ Index loaded: 5 files to process


100%|██████████| 5/5 [00:00<00:00, 59.48it/s]

Converted column 'DATE_CREATION' to datetime.
Converted column 'DATE_UPDATE' to datetime.
Columns dropped (with > 60% missing values): []
Shape of DataFrame after dropping columns: (7443, 12)


✅ Final combined CSV saved: global_tech_5_floats_full.csv


In [171]:
df = pd.read_csv("global_tech_5_floats_full.csv")
df = df.drop_duplicates()
df.head()

,N_TECH_PARAM,DATE_CREATION,DATE_UPDATE,PLATFORM_NUMBER,DATA_CENTRE,DATA_TYPE,FORMAT_VERSION,HANDBOOK_VERSION,TECHNICAL_PARAMETER_NAME,TECHNICAL_PARAMETER_VALUE,CYCLE_NUMBER,file_name
0,0,2015-10-12 09:27:39,2015-10-12 09:27:39,1900121,IN,Argo technical data,3.1,1.2,VOLTAGE_BatteryInitialAtProfileDepth_volts,14.9,1.0,1900121_tech.nc
1,1,2015-10-12 09:27:39,2015-10-12 09:27:39,1900121,IN,Argo technical data,3.1,1.2,PRESSURE_InternalVacuum_inHg,6.793,1.0,1900121_tech.nc
2,2,2015-10-12 09:27:39,2015-10-12 09:27:39,1900121,IN,Argo technical data,3.1,1.2,FLAG_ProfileTermination_hex,0,1.0,1900121_tech.nc
3,3,2015-10-12 09:27:39,2015-10-12 09:27:39,1900121,IN,Argo technical data,3.1,1.2,PRES_SurfaceOffsetTruncatedPlus5dbar_dbar,6.2,1.0,1900121_tech.nc
4,4,2015-10-12 09:27:39,2015-10-12 09:27:39,1900121,IN,Argo technical data,3.1,1.2,POSITION_PistonSurface_COUNT,205,1.0,1900121_tech.nc


In [172]:
df.shape

(7443, 12)

In [173]:
import os
import pandas as pd
import xarray as xr
import requests
from tqdm import tqdm

# === CONFIG ===
BASE_URL = "https://data-argo.ifremer.fr/dac/"  # DAC base URL including /dac/
INDEX_FILE = "/content/subset_index_csv/ar_index_global_traj_subset.csv"  # subset index CSV for 5 floats
OUTPUT_FOLDER = "global_traj_nc"                # folder to save downloaded NetCDF files
OUTPUT_CSV = "global_traj_5_floats_full.csv"   # final combined CSV

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# === FUNCTION: Download NetCDF ===
def download_nc(file_path, save_dir):
    url = BASE_URL + file_path
    local_path = os.path.join(save_dir, os.path.basename(file_path))

    if os.path.exists(local_path):
        return local_path

    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(r.content)
        return local_path
    except Exception as e:
        print(f"⚠️ Failed {file_path}: {e}")
        return None

# === FUNCTION: Convert NetCDF to DataFrame (keep all columns) ===
def nc_to_df(nc_file):
    try:
        ds = xr.open_dataset(nc_file)
        df = ds.to_dataframe().reset_index()
        ds.close()
        df["file_name"] = os.path.basename(nc_file)  # Keep file name for reference
        return df
    except Exception as e:
        print(f"⚠️ Could not read {nc_file}: {e}")
        return pd.DataFrame()  # empty DataFrame if read fails

# === STEP 1: Load index CSV ===
df_index = pd.read_csv(INDEX_FILE)
print(f"✅ Index loaded: {len(df_index)} files to process")

# === STEP 2: Loop through each file, download and convert ===
all_dfs = []

for i, row in tqdm(df_index.iterrows(), total=len(df_index)):
    nc_file = download_nc(row["file"], OUTPUT_FOLDER)
    if nc_file:
        df = nc_to_df(nc_file)
        if not df.empty:
            all_dfs.append(df)

# === STEP 3: Optional cleaning functions ===
def clean_object_columns(df):
    """Clean object/string columns"""
    object_cols = df.select_dtypes(include='object').columns
    for col in object_cols:
        df[col] = df[col].astype(str).str.replace("b'", "", regex=False)\
                                    .str.replace("'", "", regex=False)\
                                    .str.strip()
    return df

def format_date_columns(df):
    """Convert date columns to datetime objects"""
    for col in df.columns:
        if 'date' in col.lower() or 'juld' in col.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"Converted column '{col}' to datetime.")
    return df

def drop_cols_with_high_nulls(df, threshold=50):
    """
    Drops columns from a DataFrame that have a percentage of missing values
    above a specified threshold.

    Args:
        df: The pandas DataFrame to process.
        threshold: The percentage of missing values above which a column will be dropped (default is 50).

    Returns:
        The DataFrame with columns dropped.
    """
    # Calculate the percentage of missing values for each column
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    # Get the list of columns to drop
    cols_to_drop = missing_percentage[missing_percentage > threshold].index.tolist()

    # Drop the columns from the DataFrame
    df_dropped = df.drop(columns=cols_to_drop)

    print(f"Columns dropped (with > {threshold}% missing values): {cols_to_drop}")
    print(f"Shape of DataFrame after dropping columns: {df_dropped.shape}")

    return df_dropped

# === STEP 4: Combine all into single CSV ===
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df = clean_object_columns(final_df)
    final_df = final_df.drop_duplicates()
    final_df = format_date_columns(final_df)
    final_df = drop_cols_with_high_nulls(final_df, threshold=60)
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Final combined CSV saved: {OUTPUT_CSV}")
else:
    print("⚠️ No data was converted!")


✅ Index loaded: 2 files to process


  0%|          | 0/2 [00:00<?, ?it/s]/tmp/ipython-input-2661923218.py:36: FutureWarning: In a future version, xarray will not decode the variable 'CLOCK_OFFSET' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(nc_file)
100%|██████████| 2/2 [00:03<00:00,  1.91s/it]


Converted column 'DATE_CREATION' to datetime.
Converted column 'DATE_UPDATE' to datetime.
Converted column 'REFERENCE_DATE_TIME' to datetime.
Converted column 'JULD' to datetime.


/tmp/ipython-input-2661923218.py:73: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
/tmp/ipython-input-2661923218.py:73: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


Converted column 'JULD_STATUS' to datetime.
Converted column 'JULD_QC' to datetime.
Converted column 'JULD_ADJUSTED' to datetime.
Converted column 'JULD_ADJUSTED_STATUS' to datetime.
Converted column 'JULD_ADJUSTED_QC' to datetime.
Converted column 'JULD_ASCENT_START' to datetime.
Converted column 'JULD_ASCENT_START_STATUS' to datetime.
Converted column 'JULD_ASCENT_END' to datetime.
Converted column 'JULD_ASCENT_END_STATUS' to datetime.
Converted column 'JULD_DESCENT_START' to datetime.
Converted column 'JULD_DESCENT_START_STATUS' to datetime.
Converted column 'JULD_DESCENT_END' to datetime.


/tmp/ipython-input-2661923218.py:73: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
/tmp/ipython-input-2661923218.py:73: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
/tmp/ipython-input-2661923218.py:73: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
/tmp/ipython-input-2661923218.py:73: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent

Converted column 'JULD_DESCENT_END_STATUS' to datetime.
Converted column 'JULD_TRANSMISSION_START' to datetime.
Converted column 'JULD_TRANSMISSION_START_STATUS' to datetime.
Converted column 'JULD_FIRST_STABILIZATION' to datetime.
Converted column 'JULD_FIRST_STABILIZATION_STATUS' to datetime.
Converted column 'JULD_PARK_START' to datetime.
Converted column 'JULD_PARK_START_STATUS' to datetime.
Converted column 'JULD_PARK_END' to datetime.
Converted column 'JULD_PARK_END_STATUS' to datetime.
Converted column 'JULD_DEEP_PARK_START' to datetime.
Converted column 'JULD_DEEP_PARK_START_STATUS' to datetime.
Converted column 'JULD_DEEP_DESCENT_END' to datetime.
Converted column 'JULD_DEEP_DESCENT_END_STATUS' to datetime.
Converted column 'JULD_DEEP_ASCENT_START' to datetime.
Converted column 'JULD_DEEP_ASCENT_START_STATUS' to datetime.
Converted column 'JULD_TRANSMISSION_END' to datetime.
Converted column 'JULD_TRANSMISSION_END_STATUS' to datetime.
Converted column 'JULD_FIRST_MESSAGE' to d

/tmp/ipython-input-2661923218.py:73: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


Columns dropped (with > 60% missing values): ['JULD_STATUS', 'JULD_QC', 'JULD_ADJUSTED', 'JULD_ADJUSTED_STATUS', 'JULD_ADJUSTED_QC', 'CYCLE_NUMBER_ADJUSTED', 'PRES', 'PRES_ADJUSTED', 'PRES_ADJUSTED_ERROR', 'TEMP', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_ERROR', 'PSAL', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_ERROR', 'AXES_ERROR_ELLIPSE_MAJOR', 'AXES_ERROR_ELLIPSE_MINOR', 'AXES_ERROR_ELLIPSE_ANGLE', 'JULD_ASCENT_START_STATUS', 'JULD_ASCENT_END_STATUS', 'JULD_DESCENT_START_STATUS', 'JULD_DESCENT_END', 'JULD_DESCENT_END_STATUS', 'JULD_TRANSMISSION_START_STATUS', 'JULD_FIRST_STABILIZATION', 'JULD_FIRST_STABILIZATION_STATUS', 'JULD_PARK_START_STATUS', 'JULD_PARK_END', 'JULD_PARK_END_STATUS', 'JULD_DEEP_PARK_START', 'JULD_DEEP_PARK_START_STATUS', 'JULD_DEEP_DESCENT_END', 'JULD_DEEP_DESCENT_END_STATUS', 'JULD_DEEP_ASCENT_START', 'JULD_DEEP_ASCENT_START_STATUS', 'JULD_TRANSMISSION_END_STATUS', 'JULD_FIRST_MESSAGE_STATUS', 'JULD_FIRST_LOCATION_STATUS', 'JULD_LAST_MESSAGE_STATUS', 'JULD_LAST_LOCATION_STATUS', '

In [182]:
import pandas as pd

# Load the CSV file with low_memory=False to avoid the DtypeWarning
df = pd.read_csv("global_traj_5_floats_full.csv", low_memory=False)

# Display the head to confirm it loaded
print(df.head())

   N_PARAM  N_MEASUREMENT  N_CYCLE  N_HISTORY        DATE_CREATION  \
0      0.0              0        0          0  2016-06-23 03:20:57   
1      0.0              0        1          0  2016-06-23 03:20:57   
2      0.0              0        2          0  2016-06-23 03:20:57   
3      0.0              0        3          0  2016-06-23 03:20:57   
4      0.0              0        4          0  2016-06-23 03:20:57   

           DATE_UPDATE  PLATFORM_NUMBER DATA_CENTRE  WMO_INST_TYPE  \
0  2016-06-23 08:51:37          1900121          IN            846   
1  2016-06-23 08:51:37          1900121          IN            846   
2  2016-06-23 08:51:37          1900121          IN            846   
3  2016-06-23 08:51:37          1900121          IN            846   
4  2016-06-23 08:51:37          1900121          IN            846   

  PROJECT_NAME  ...          JULD_TRANSMISSION_END  \
0   Argo INDIA  ...  2002-11-11 22:28:29.045715712   
1   Argo INDIA  ...  2002-11-21 22:28:25.070158848

In [183]:
# Check for missing values in each column
missing_values = df.isnull().sum()

# Display columns with missing values
print("Missing values per column:")
print(missing_values[missing_values > 0])

Missing values per column:
N_PARAM                     42198
TRAJECTORY_PARAMETERS       42198
PLATFORM_TYPE               42198
FLOAT_SERIAL_NO             42198
FIRMWARE_VERSION            42198
JULD                       176715
LATITUDE                   294030
LONGITUDE                  294030
POSITION_ACCURACY          294327
POSITION_QC                294030
MEASUREMENT_CODE            42198
JULD_ASCENT_START           14066
JULD_ASCENT_END             14066
JULD_DESCENT_START          13525
JULD_TRANSMISSION_START     42198
JULD_PARK_START             42198
JULD_TRANSMISSION_END       42198
JULD_FIRST_MESSAGE          42198
JULD_FIRST_LOCATION         42198
JULD_LAST_MESSAGE           42198
JULD_LAST_LOCATION          42198
GROUNDED                    14066
CONFIG_MISSION_NUMBER       42198
CYCLE_NUMBER_INDEX          42198
dtype: int64


In [184]:
df.shape

(590757, 43)

In [185]:
df = df.drop_duplicates()
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Final combined CSV saved: {OUTPUT_CSV}")

✅ Final combined CSV saved: global_traj_5_floats_full.csv


In [186]:
df.columns

Index(['N_PARAM', 'N_MEASUREMENT', 'N_CYCLE', 'N_HISTORY', 'DATE_CREATION',
       'DATE_UPDATE', 'PLATFORM_NUMBER', 'DATA_CENTRE', 'WMO_INST_TYPE',
       'PROJECT_NAME', 'PI_NAME', 'DATA_TYPE', 'FORMAT_VERSION',
       'HANDBOOK_VERSION', 'REFERENCE_DATE_TIME', 'POSITIONING_SYSTEM',
       'TRAJECTORY_PARAMETERS', 'DATA_STATE_INDICATOR', 'PLATFORM_TYPE',
       'FLOAT_SERIAL_NO', 'FIRMWARE_VERSION', 'JULD', 'LATITUDE', 'LONGITUDE',
       'POSITION_ACCURACY', 'POSITION_QC', 'CYCLE_NUMBER', 'MEASUREMENT_CODE',
       'JULD_ASCENT_START', 'JULD_ASCENT_END', 'JULD_DESCENT_START',
       'JULD_TRANSMISSION_START', 'JULD_PARK_START', 'JULD_TRANSMISSION_END',
       'JULD_FIRST_MESSAGE', 'JULD_FIRST_LOCATION', 'JULD_LAST_MESSAGE',
       'JULD_LAST_LOCATION', 'GROUNDED', 'CONFIG_MISSION_NUMBER',
       'CYCLE_NUMBER_INDEX', 'DATA_MODE', 'file_name'],
      dtype='object')

In [192]:
import pandas as pd

# Load the full CSV
full_csv = "global_traj_5_floats_full.csv"
df = pd.read_csv(full_csv)
df = df.drop_duplicates()

important_cols_traj = [
    "PLATFORM_NUMBER",      # Unique float ID
    "FLOAT_SERIAL_NO",      # Serial number of the float
    "PLATFORM_TYPE",        # Type of float/platform
    "DATA_CENTRE",          # DAC providing the data
    "PROJECT_NAME",         # Project/funding info
    "PI_NAME",              # Principal investigator
    "POSITIONING_SYSTEM",   # GPS / Argos / other system
    "DATA_STATE_INDICATOR", # Real-time, adjusted, etc.
    "JULD",                 # Timestamp (Julian days)
    "LATITUDE",             # Latitude of float
    "LONGITUDE",            # Longitude of float
    "POSITION_QC",          # Position quality control
    "POSITION_ACCURACY",    # Accuracy estimate of position
    "CYCLE_NUMBER",         # Float cycle number
    "DATA_MODE",            # Mode of data (R = real-time, A = adjusted)
    "file_name"             # Source NetCDF filename
]


# Keep only the columns that exist in the dataframe
cols_to_keep = [c for c in important_cols if c in df.columns]
df = df[cols_to_keep]

/tmp/ipython-input-4035512532.py:5: DtypeWarning: Columns (16,18,31,32,33,34,35,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(full_csv)


In [189]:
df_imp = df_imp.drop_duplicates()

In [193]:
df.shape

(590757, 7)

In [194]:


# Save filtered CSV

df_imp.to_csv("global_traj_5_floats.csv", index=False)
print(f"✅ Important columns CSV saved: global_traj_5_floats.csv")
print(df_imp.shape)

✅ Important columns CSV saved: global_traj_5_floats.csv
(2, 7)
